# FarmSight XAI Project Setup
This notebook handles the initial workspace environment setup, including mounting Google Drive, configuring directory paths, and documenting key library dependencies.

In [1]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# Define paths
base_path = '/content/drive/MyDrive/FarmSight_XAI'
folders_to_create = [
    os.path.join(base_path, 'data/raw'),
    os.path.join(base_path, 'data/interim'),
    os.path.join(base_path, 'data/processed'),
    os.path.join(base_path, 'models'),
    os.path.join(base_path, 'results/figures'),
    os.path.join(base_path, 'results/tables'),
    os.path.join(base_path, 'logs')
]

# 2. & 3. Create folder structure and print every created folder path
print("Creating project directory structure:")
for folder in folders_to_create:
    os.makedirs(folder, exist_ok=True)
    print(f"Created or verified: {folder}")

Mounted at /content/drive
Creating project directory structure:
Created or verified: /content/drive/MyDrive/FarmSight_XAI/data/raw
Created or verified: /content/drive/MyDrive/FarmSight_XAI/data/interim
Created or verified: /content/drive/MyDrive/FarmSight_XAI/data/processed
Created or verified: /content/drive/MyDrive/FarmSight_XAI/models
Created or verified: /content/drive/MyDrive/FarmSight_XAI/results/figures
Created or verified: /content/drive/MyDrive/FarmSight_XAI/results/tables
Created or verified: /content/drive/MyDrive/FarmSight_XAI/logs


In [2]:
# 4. Print Python version
python_version = sys.version
print(f"Python Version: {python_version}")

# 5. Retrieve currently installed versions of required libraries
libraries = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'xgboost', 'shap', 'lime']
versions = {}

for lib in libraries:
    # Map import names to pip packaging name differences where appropriate
    import_name = 'sklearn' if lib == 'scikit-learn' else lib
    try:
        imported_lib = __import__(import_name)
        versions[lib] = getattr(imported_lib, '__version__', 'installed')
    except ImportError:
        versions[lib] = 'not installed'

# Save the information to environment_versions.txt
env_file_path = os.path.join(base_path, 'logs/environment_versions.txt')
with open(env_file_path, 'w') as f:
    f.write(f"Python version: {python_version}\n")
    for lib, ver in versions.items():
        f.write(f"{lib}: {ver}\n")

print(f"\nSaved environment metadata to: {env_file_path}")

Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

Saved environment metadata to: /content/drive/MyDrive/FarmSight_XAI/logs/environment_versions.txt


In [3]:
# Validation
all_folders_exist = all(os.path.exists(folder) for folder in folders_to_create)
file_exists = os.path.exists(env_file_path)

print("--- Validation Summary ---")
print(f"All directories exist: {all_folders_exist}")
print(f"Environment file exists: {file_exists}")

if all_folders_exist and file_exists:
    print("\nSETUP VERIFIED")
else:
    print("\nSETUP FAILED - Check output logs above.")

--- Validation Summary ---
All directories exist: True
Environment file exists: True

SETUP VERIFIED


## Data Acquisition & Verification Stage
This section manages downloading the World Bank agricultural dataset, extracting the designated ICRISAT apportioned crop data, verifying its properties, and documenting data provenance.

In [4]:
import requests
import hashlib
import zipfile
import os

# Configure URL and paths
zip_url = "https://reproducibility.worldbank.org/catalog/409/download/1152/RR_IND_2025_488.zip"
temp_zip_path = "/content/RR_IND_2025_488.zip"

# 1. & 2. Download the ZIP package with streaming and verify HTTP status
print(f"Downloading {zip_url} ...")
response = requests.get(zip_url, stream=True)
if response.status_code != 200:
    raise RuntimeError(f"Download failed with HTTP status code: {response.status_code}")

# 3. Save the downloaded ZIP temporarily under /content/
with open(temp_zip_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

# 4. Print size in MB and calculate SHA-256 checksum
file_size_mb = os.path.getsize(temp_zip_path) / (1024 * 1024)
print(f"Downloaded ZIP Size: {file_size_mb:.2f} MB")

sha256_hash = hashlib.sha256()
with open(temp_zip_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        sha256_hash.update(byte_block)
zip_sha256 = sha256_hash.hexdigest()
print(f"ZIP SHA-256 Checksum: {zip_sha256}")

Downloaded ZIP Size: 329.80 MB
ZIP SHA-256 Checksum: f9ff83879f0065a7563aedccdb4590fa224eb1c75519cc725c35a9ae5f5d18cd


In [5]:
# 5. & 6. Inspect ZIP contents without fully extracting and search for the target file
target_suffix = "Crops_Area Production Yield.csv"
target_path_sub = "raw/DLD/Apportioned/"
target_zip_path = None

with zipfile.ZipFile(temp_zip_path, 'r') as zip_ref:
    for file_info in zip_ref.infolist():
        if file_info.filename.endswith(target_suffix) and target_path_sub in file_info.filename:
            target_zip_path = file_info.filename
            break

if not target_zip_path:
    raise FileNotFoundError("Specified target apportioned crop CSV file was not found inside the ZIP package.")

print(f"Found target CSV inside ZIP: {target_zip_path}")

# 7. & 8. Extract only that CSV and save to raw data directory
destination_csv_path = "/content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv"

with zipfile.ZipFile(temp_zip_path, 'r') as zip_ref:
    with zip_ref.open(target_zip_path) as source_file:
        with open(destination_csv_path, 'wb') as dest_file:
            dest_file.write(source_file.read())

print(f"Extracted and saved to: {destination_csv_path}")

Found target CSV inside ZIP: RR_IND_2025_488/Reproducbility Package/raw/DLD/Apportioned/Crops_Area Production Yield.csv
Extracted and saved to: /content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv


In [6]:
import pandas as pd
from datetime import datetime, timezone

# 10. Load the saved CSV only for verification
df = pd.read_csv(destination_csv_path)

# Calculate CSV SHA-256
csv_hash = hashlib.sha256()
with open(destination_csv_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        csv_hash.update(byte_block)
csv_sha256 = csv_hash.hexdigest()

# 11. Print required data insights
csv_size_mb = os.path.getsize(destination_csv_path) / (1024 * 1024)
row_count, col_count = df.shape

print(f"Saved Path: {destination_csv_path}")
print(f"File Size: {csv_size_mb:.2f} MB")
print(f"Row Count: {row_count}")
print(f"Column Count: {col_count}")
print("\nComplete Column-Name List:")
print(df.columns.tolist())
print("\nFirst 5 Rows:")
display(df.head())
print("\nMissing-Value Count for Each Column:")
print(df.isnull().sum())

Saved Path: /content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv
File Size: 8.83 MB
Row Count: 16146
Column Count: 80

Complete Column-Name List:
['Dist Code', 'Year', 'State Code', 'State Name', 'Dist Name', 'RICE AREA (1000 ha)', 'RICE PRODUCTION (1000 tons)', 'RICE YIELD (Kg per ha)', 'WHEAT AREA (1000 ha)', 'WHEAT PRODUCTION (1000 tons)', 'WHEAT YIELD (Kg per ha)', 'KHARIF SORGHUM AREA (1000 ha)', 'KHARIF SORGHUM PRODUCTION (1000 tons)', 'KHARIF SORGHUM YIELD (Kg per ha)', 'RABI SORGHUM AREA (1000 ha)', 'RABI SORGHUM PRODUCTION (1000 tons)', 'RABI SORGHUM YIELD (Kg per ha)', 'SORGHUM AREA (1000 ha)', 'SORGHUM PRODUCTION (1000 tons)', 'SORGHUM YIELD (Kg per ha)', 'PEARL MILLET AREA (1000 ha)', 'PEARL MILLET PRODUCTION (1000 tons)', 'PEARL MILLET YIELD (Kg per ha)', 'MAIZE AREA (1000 ha)', 'MAIZE PRODUCTION (1000 tons)', 'MAIZE YIELD (Kg per ha)', 'FINGER MILLET AREA (1000 ha)', 'FINGER MILLET PRODUCTION (1000 tons)', 'FINGER MILLET YI

,Dist Code,Year,State Code,State Name,Dist Name,RICE AREA (1000 ha),RICE PRODUCTION (1000 tons),RICE YIELD (Kg per ha),WHEAT AREA (1000 ha),WHEAT PRODUCTION (1000 tons),...,SUGARCANE YIELD (Kg per ha),COTTON AREA (1000 ha),COTTON PRODUCTION (1000 tons),COTTON YIELD (Kg per ha),FRUITS AREA (1000 ha),VEGETABLES AREA (1000 ha),FRUITS AND VEGETABLES AREA (1000 ha),POTATOES AREA (1000 ha),ONION AREA (1000 ha),FODDER AREA (1000 ha)
0,1,1966,14,Chhattisgarh,Durg,548.0,185.0,337.59,44.0,20.0,...,1777.78,0.0,0.0,0.0,5.95,6.64,12.59,0.01,0.60,0.47
1,1,1967,14,Chhattisgarh,Durg,547.0,409.0,747.71,50.0,26.0,...,1500.00,0.0,0.0,0.0,5.77,7.24,13.02,0.01,0.56,1.23
2,1,1968,14,Chhattisgarh,Durg,556.3,468.0,841.27,53.7,30.0,...,1000.00,0.0,0.0,0.0,5.41,7.40,12.81,0.10,0.58,1.02
3,1,1969,14,Chhattisgarh,Durg,563.4,400.8,711.40,49.4,26.5,...,1900.00,0.0,0.0,0.0,5.52,7.16,12.69,0.01,0.56,0.84
4,1,1970,14,Chhattisgarh,Durg,571.6,473.6,828.55,44.2,29.0,...,2000.00,0.0,0.0,0.0,5.45,7.19,12.64,0.02,0.52,0.42



Missing-Value Count for Each Column:
Dist Code                               0
Year                                    0
State Code                              0
State Name                              0
Dist Name                               0
                                       ..
VEGETABLES AREA (1000 ha)               0
FRUITS AND VEGETABLES AREA (1000 ha)    0
POTATOES AREA (1000 ha)                 0
ONION AREA (1000 ha)                    0
FODDER AREA (1000 ha)                   0
Length: 80, dtype: int64


In [7]:
# 12. Create icrisat_data_provenance.txt
provenance_file_path = "/content/drive/MyDrive/FarmSight_XAI/logs/icrisat_data_provenance.txt"
current_utc_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

provenance_content = f"""Original Package URL: {zip_url}
Catalog Page URL: https://reproducibility.worldbank.org/index.php/catalog/409
Original Internal ZIP Path: {target_zip_path}
Extraction Date & Time: {current_utc_time}
Extracted CSV Filename: icrisat_dld_crops_area_production_yield_apportioned.csv
Extracted CSV SHA-256 Checksum: {csv_sha256}
Row Count: {row_count}
Column Count: {col_count}
Stated Source: ICRISAT and Tata-Cornell Institute District Level Database
Access Route: World Bank reproducibility package
"""

with open(provenance_file_path, 'w') as f:
    f.write(provenance_content)

print(f"Provenance logged successfully to: {provenance_file_path}")

Provenance logged successfully to: /content/drive/MyDrive/FarmSight_XAI/logs/icrisat_data_provenance.txt


In [8]:
# Validation checks
csv_exists = os.path.exists(destination_csv_path)
prov_exists = os.path.exists(provenance_file_path)
valid_dims = False

if csv_exists:
    # Read minimal layout to check structure
    test_df = pd.read_csv(destination_csv_path, nrows=1)
    valid_dims = test_df.shape[0] >= 0 and test_df.shape[1] >= 1 # At least one row and one column setup checked

print("\n--- Acquisition Validation Summary ---")
print(f"CSV File Exists: {csv_exists}")
print(f"CSV Contains Rows/Cols: {valid_dims}")
print(f"Provenance File Exists: {prov_exists}")

if csv_exists and valid_dims and prov_exists:
    print("\nICRISAT ACQUISITION VERIFIED")
else:
    print("\nACQUISITION VALIDATION FAILED")


--- Acquisition Validation Summary ---
CSV File Exists: True
CSV Contains Rows/Cols: True
Provenance File Exists: True

ICRISAT ACQUISITION VERIFIED


## Read-Only Dataset Audit Stage
This section performs a descriptive audit of the raw ICRISAT dataset, saves a structural inventory, and writes a metadata summary log without modifying the original data.

In [9]:
import os
import pandas as pd
import numpy as np

raw_path = "/content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv"

# 1. Load the CSV without changing original file
df_audit = pd.read_csv(raw_path)

# 2. Print exact shape, column list, data types, memory, duplicates and missing metrics
print(f"Dataset Shape: {df_audit.shape}")
print(f"Memory Usage (Total): {df_audit.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print(f"Number of Exact Duplicate Rows: {df_audit.duplicated().sum()}")

print("\n--- Column Data Types & Missing-Value Statistics ---")
missing_counts = df_audit.isnull().sum()
missing_pct = (missing_counts / len(df_audit)) * 100

audit_cols_df = pd.DataFrame({
    'Dtype': df_audit.dtypes,
    'Missing_Count': missing_counts,
    'Missing_Percentage': missing_pct
})
pd.set_option('display.max_rows', 100)
display(audit_cols_df)

Dataset Shape: (16146, 80)
Memory Usage (Total): 11.41 MB
Number of Exact Duplicate Rows: 0

--- Column Data Types & Missing-Value Statistics ---


,Dtype,Missing_Count,Missing_Percentage
Dist Code,int64,0,0.0
Year,int64,0,0.0
State Code,int64,0,0.0
State Name,object,0,0.0
Dist Name,object,0,0.0
RICE AREA (1000 ha),float64,0,0.0
RICE PRODUCTION (1000 tons),float64,0,0.0
RICE YIELD (Kg per ha),float64,0,0.0
WHEAT AREA (1000 ha),float64,0,0.0
WHEAT PRODUCTION (1000 tons),float64,0,0.0


In [10]:
# 3. Identify and print identifier columns
id_cols = {
    'district_code': 'Dist Code',
    'district_name': 'Dist Name',
    'state_code': 'State Code',
    'state_name': 'State Name',
    'year': 'Year'
}
print("Identifier Columns map:", id_cols)

# 4. Temporal and spatial span metrics
min_year = int(df_audit['Year'].min())
max_year = int(df_audit['Year'].max())
num_years = df_audit['Year'].nunique()
num_states = df_audit['State Name'].nunique()
num_districts = df_audit['Dist Name'].nunique() # Unique names
num_dist_codes = df_audit['Dist Code'].nunique() # Unique codes

print(f"Minimum Year: {min_year}")
print(f"Maximum Year: {max_year}")
print(f"Number of Unique Years: {num_years}")
print(f"Number of Unique States: {num_states}")
print(f"Number of Unique Districts (by Name): {num_districts}")
print(f"Number of Unique Districts (by Code): {num_dist_codes}")

Identifier Columns map: {'district_code': 'Dist Code', 'district_name': 'Dist Name', 'state_code': 'State Code', 'state_name': 'State Name', 'year': 'Year'}
Minimum Year: 1966
Maximum Year: 2017
Number of Unique Years: 52
Number of Unique States: 20
Number of Unique Districts (by Name): 311
Number of Unique Districts (by Code): 311


In [11]:
# 5. List all columns containing AREA, PRODUCTION or YIELD in their names
metric_cols = [c for c in df_audit.columns if any(kw in c for kw in ['AREA', 'PRODUCTION', 'YIELD'])]
print(f"Found {len(metric_cols)} metric columns.")

# 6. For each crop-yield column, count: non-missing, zero, negative values
yield_cols = [c for c in df_audit.columns if 'YIELD' in c]
yield_stats = []

for col in yield_cols:
    non_missing = df_audit[col].notnull().sum()
    zeros = (df_audit[col] == 0).sum()
    negatives = (df_audit[col] < 0).sum()
    yield_stats.append({
        'Column': col,
        'Non_Missing': non_missing,
        'Zeros': zeros,
        'Negatives': negatives
    })

yield_stats_df = pd.DataFrame(yield_stats)
display(yield_stats_df)

Found 75 metric columns.


,Column,Non_Missing,Zeros,Negatives
0,RICE YIELD (Kg per ha),16146,1143,22
1,WHEAT YIELD (Kg per ha),16146,2369,35
2,KHARIF SORGHUM YIELD (Kg per ha),16146,4864,40
3,RABI SORGHUM YIELD (Kg per ha),16146,12113,40
4,SORGHUM YIELD (Kg per ha),16146,4541,40
5,PEARL MILLET YIELD (Kg per ha),16146,6133,54
6,MAIZE YIELD (Kg per ha),16146,1964,28
7,FINGER MILLET YIELD (Kg per ha),16146,10243,20
8,BARLEY YIELD (Kg per ha),16146,8198,20
9,CHICKPEA YIELD (Kg per ha),16146,2235,42


In [12]:
# 7. Print the first 5 rows using identifier columns + Rice, Maize and Cotton area, prod, yield
target_keywords = ['RICE', 'MAIZE', 'COTTON']
selected_metrics = [c for c in df_audit.columns if any(k in c for k in target_keywords)]
print_cols = list(id_cols.values()) + selected_metrics

print("Displaying subset of target crops (Rice, Maize, Cotton):")
display(df_audit[print_cols].head(5))

Displaying subset of target crops (Rice, Maize, Cotton):


,Dist Code,Dist Name,State Code,State Name,Year,RICE AREA (1000 ha),RICE PRODUCTION (1000 tons),RICE YIELD (Kg per ha),MAIZE AREA (1000 ha),MAIZE PRODUCTION (1000 tons),MAIZE YIELD (Kg per ha),COTTON AREA (1000 ha),COTTON PRODUCTION (1000 tons),COTTON YIELD (Kg per ha)
0,1,Durg,14,Chhattisgarh,1966,548.0,185.0,337.59,3.0,2.0,666.67,0.0,0.0,0.0
1,1,Durg,14,Chhattisgarh,1967,547.0,409.0,747.71,3.0,3.0,1000.00,0.0,0.0,0.0
2,1,Durg,14,Chhattisgarh,1968,556.3,468.0,841.27,2.8,2.0,714.29,0.0,0.0,0.0
3,1,Durg,14,Chhattisgarh,1969,563.4,400.8,711.40,2.7,2.3,851.85,0.0,0.0,0.0
4,1,Durg,14,Chhattisgarh,1970,571.6,473.6,828.55,2.5,3.3,1320.00,0.0,0.0,0.0


In [13]:
# 8. Save a complete column inventory to: results/tables/icrisat_column_inventory.csv
inventory_path = "/content/drive/MyDrive/FarmSight_XAI/results/tables/icrisat_column_inventory.csv"
audit_cols_df.reset_index().rename(columns={'index': 'Column_Name'}).to_csv(inventory_path, index=False)
print(f"Column inventory successfully written to: {inventory_path}")

Column inventory successfully written to: /content/drive/MyDrive/FarmSight_XAI/results/tables/icrisat_column_inventory.csv


In [14]:
# 9. Save the audit summary to: logs/icrisat_raw_audit.txt
audit_log_path = "/content/drive/MyDrive/FarmSight_XAI/logs/icrisat_raw_audit.txt"

audit_summary_text = f"""FarmSight XAI ICRISAT Raw Data Audit
=========================================
Dataset Path: {raw_path}
Shape: {df_audit.shape[0]} rows, {df_audit.shape[1]} columns
Total Memory Usage: {df_audit.memory_usage(deep=True).sum() / (1024**2):.2f} MB
Duplicate Rows: {df_audit.duplicated().sum()}

Temporal & Spatial Span:
------------------------
Year Range: {min_year} - {max_year} ({num_years} unique years)
Unique States: {num_states}
Unique Districts (by Code): {num_dist_codes}

Metric Breakdown:
-----------------
Total Area/Production/Yield Columns: {len(metric_cols)}
Total Yield Columns Analyzed: {len(yield_cols)}
"""

with open(audit_log_path, 'w') as f:
    f.write(audit_summary_text)
    f.write("\nYield Columns Statistics:\n")
    f.write(yield_stats_df.to_string(index=False))

print(f"Audit summary written to: {audit_log_path}")

Audit summary written to: /content/drive/MyDrive/FarmSight_XAI/logs/icrisat_raw_audit.txt


In [15]:
# Validation checks
import hashlib

# Confirm the raw CSV is structurally identical to the loaded context (not modified)
original_hash_expected = '7857decb1a196a54aac71a0f841af43c3bac98b3b90744aadca18b1859cea6f3'
current_raw_hash = hashlib.sha256()
with open(raw_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        current_raw_hash.update(byte_block)
raw_unmodified = (current_raw_hash.hexdigest() == original_hash_expected)

# Confirm file generation
inventory_exists = os.path.exists(inventory_path) and os.path.getsize(inventory_path) > 0
audit_log_exists = os.path.exists(audit_log_path) and os.path.getsize(audit_log_path) > 0

print("\n--- Audit Verification Summary ---")
print(f"Raw CSV File Unmodified: {raw_unmodified}")
print(f"Inventory CSV Created: {inventory_exists}")
print(f"Audit Log Created: {audit_log_exists}")

if raw_unmodified and inventory_exists and audit_log_exists:
    print("\nICRISAT RAW AUDIT VERIFIED")
else:
    print("\nAUDIT VERIFICATION FAILED")


--- Audit Verification Summary ---
Raw CSV File Unmodified: True
Inventory CSV Created: True
Audit Log Created: True

ICRISAT RAW AUDIT VERIFIED


## Candidate Crop Coverage Analysis Stage
This section conducts a targeted read-only audit of spatial and temporal coverage for Rice, Maize, and Cotton to assist in scoping our experiments.

In [16]:
import os
import pandas as pd

raw_path = "/content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv"
df_cov = pd.read_csv(raw_path)

candidates = ['RICE', 'MAIZE', 'COTTON']
crop_cols = {}

# 1. Identify the exact columns using case-insensitive match
for crop in candidates:
    crop_cols[crop] = {
        'area': [c for c in df_cov.columns if crop in c.upper() and 'AREA' in c.upper()][0],
        'production': [c for c in df_cov.columns if crop in c.upper() and 'PRODUCTION' in c.upper()][0],
        'yield': [c for c in df_cov.columns if crop in c.upper() and 'YIELD' in c.upper()][0]
    }

print("Matched Columns:")
for crop, cols in crop_cols.items():
    print(f"{crop}: {cols}")

Matched Columns:
RICE: {'area': 'RICE AREA (1000 ha)', 'production': 'RICE PRODUCTION (1000 tons)', 'yield': 'RICE YIELD (Kg per ha)'}
MAIZE: {'area': 'MAIZE AREA (1000 ha)', 'production': 'MAIZE PRODUCTION (1000 tons)', 'yield': 'MAIZE YIELD (Kg per ha)'}
COTTON: {'area': 'COTTON AREA (1000 ha)', 'production': 'COTTON PRODUCTION (1000 tons)', 'yield': 'COTTON YIELD (Kg per ha)'}


In [17]:
# 2. & 3. Calculate coverage and era-wise breakdowns
coverage_stats = []
era_bins = [
    (1966, 1989, "1966-1989"),
    (1990, 1999, "1990-1999"),
    (2000, 2009, "2000-2009"),
    (2010, 2017, "2010-2017")
]

for crop in candidates:
    y_col = crop_cols[crop]['yield']

    total_rows = len(df_cov)
    non_missing_yield = df_cov[y_col].notnull().sum()
    positive_yield = (df_cov[y_col] > 0).sum()
    zero_yield = (df_cov[y_col] == 0).sum()
    negative_yield = (df_cov[y_col] < 0).sum()

    # Filtering for positive yield subset
    pos_df = df_cov[df_cov[y_col] > 0]
    earliest_year = int(pos_df['Year'].min()) if not pos_df.empty else None
    latest_year = int(pos_df['Year'].max()) if not pos_df.empty else None
    unique_states = pos_df['State Name'].nunique() if not pos_df.empty else 0
    unique_districts = pos_df['Dist Name'].nunique() if not pos_df.empty else 0

    # Era breakdown for positive yield records
    era_counts = {}
    for start, end, label in era_bins:
        era_counts[label] = ((pos_df['Year'] >= start) & (pos_df['Year'] <= end)).sum()

    stat_dict = {
        'Crop': crop.capitalize(),
        'Total_Rows': total_rows,
        'Non_Missing_Yield': non_missing_yield,
        'Positive_Yield': positive_yield,
        'Zero_Yield': zero_yield,
        'Negative_Yield': negative_yield,
        'Earliest_Pos_Year': earliest_year,
        'Latest_Pos_Year': latest_year,
        'Unique_States_Pos': unique_states,
        'Unique_Districts_Pos': unique_districts
    }
    # Merge era counts
    for label, count in era_counts.items():
        stat_dict[f'Pos_Yield_Rows_{label}'] = count

    coverage_stats.append(stat_dict)

coverage_df = pd.DataFrame(coverage_stats)
print("\n--- Candidate Crop Coverage Summary ---")
display(coverage_df)


--- Candidate Crop Coverage Summary ---


,Crop,Total_Rows,Non_Missing_Yield,Positive_Yield,Zero_Yield,Negative_Yield,Earliest_Pos_Year,Latest_Pos_Year,Unique_States_Pos,Unique_Districts_Pos,Pos_Yield_Rows_1966-1989,Pos_Yield_Rows_1990-1999,Pos_Yield_Rows_2000-2009,Pos_Yield_Rows_2010-2017
0,Rice,16146,16146,14981,1143,22,1966,2017,20,307,6952,2907,2874,2248
1,Maize,16146,16146,14154,1964,28,1966,2017,20,302,6382,2715,2797,2260
2,Cotton,16146,16146,7816,8315,15,1966,2017,19,231,3648,1525,1479,1164


In [18]:
# 5. & 6. Save outputs to results and log directory
table_path = "/content/drive/MyDrive/FarmSight_XAI/results/tables/candidate_crop_coverage.csv"
report_path = "/content/drive/MyDrive/FarmSight_XAI/logs/candidate_crop_coverage.txt"

# Save csv table
coverage_df.to_csv(table_path, index=False)
print(f"Saved comparative table to: {table_path}")

# Generate and save text report
report_content = f"""FarmSight XAI Candidate Crop Coverage Report
==============================================
Raw Dataset evaluated: {raw_path}
Target Crops: Rice, Maize, Cotton

"""
report_content += coverage_df.to_string(index=False)

with open(report_path, 'w') as f:
    f.write(report_content)
print(f"Saved text report to: {report_path}")

Saved comparative table to: /content/drive/MyDrive/FarmSight_XAI/results/tables/candidate_crop_coverage.csv
Saved text report to: /content/drive/MyDrive/FarmSight_XAI/logs/candidate_crop_coverage.txt


In [19]:
# Validation checks
import hashlib

original_hash_expected = '7857decb1a196a54aac71a0f841af43c3bac98b3b90744aadca18b1859cea6f3'
current_raw_hash = hashlib.sha256()
with open(raw_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        current_raw_hash.update(byte_block)
raw_unmodified = (current_raw_hash.hexdigest() == original_hash_expected)

table_valid = os.path.exists(table_path) and os.path.getsize(table_path) > 0
report_valid = os.path.exists(report_path) and os.path.getsize(report_path) > 0

print("\n--- Verification Summary ---")
print(f"Raw dataset hash unchanged: {raw_unmodified}")
print(f"Coverage CSV table exists & non-empty: {table_valid}")
print(f"Coverage text report exists & non-empty: {report_valid}")

if raw_unmodified and table_valid and report_valid:
    print("\nCANDIDATE CROP COVERAGE VERIFIED")
else:
    print("\nCOVERAGE VERIFICATION FAILED")


--- Verification Summary ---
Raw dataset hash unchanged: True
Coverage CSV table exists & non-empty: True
Coverage text report exists & non-empty: True

CANDIDATE CROP COVERAGE VERIFIED


In [20]:
import os
import hashlib
import pandas as pd
import numpy as np

raw_path = "/content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv"
interim_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/icrisat_rice_maize_cotton_long.csv"
conflicting_path = "/content/drive/MyDrive/FarmSight_XAI/results/tables/icrisat_conflicting_duplicates.csv"
report_path = "/content/drive/MyDrive/FarmSight_XAI/logs/icrisat_restructuring_report.txt"

# Expected raw SHA-256 to ensure read-only safety
expected_raw_hash = "7857decb1a196a54aac71a0f841af43c3bac98b3b90744aadca18b1859cea6f3"

def get_sha256(filepath):
    sha256 = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256.update(byte_block)
    return sha256.hexdigest()

# 1. Load the raw CSV
df_raw = pd.read_csv(raw_path)
raw_shape = df_raw.shape
print(f"Raw dataset loaded. Shape: {raw_shape}")

Raw dataset loaded. Shape: (16146, 80)


In [21]:
# 2. & 3. Column identifiers and crop map
id_cols = {
    'Dist Code': 'district_code',
    'Dist Name': 'district_name',
    'State Code': 'state_code',
    'State Name': 'state_name',
    'Year': 'year'
}

crop_cols_map = {
    'Rice': {
        'area': 'RICE AREA (1000 ha)',
        'production': 'RICE PRODUCTION (1000 tons)',
        'yield': 'RICE YIELD (Kg per ha)'
    },
    'Maize': {
        'area': 'MAIZE AREA (1000 ha)',
        'production': 'MAIZE PRODUCTION (1000 tons)',
        'yield': 'MAIZE YIELD (Kg per ha)'
    },
    'Cotton': {
        'area': 'COTTON AREA (1000 ha)',
        'production': 'COTTON PRODUCTION (1000 tons)',
        'yield': 'COTTON YIELD (Kg per ha)'
    }
}

# 4. & 5. Reshape wide to long
long_records = []
for index, row in df_raw.iterrows():
    base_info = {id_cols[k]: row[k] for k in id_cols}
    for crop, metrics in crop_cols_map.items():
        rec = base_info.copy()
        rec['crop'] = crop
        rec['area_1000_ha'] = row[metrics['area']]
        rec['production_1000_tons'] = row[metrics['production']]
        rec['yield_kg_per_ha'] = row[metrics['yield']]
        long_records.append(rec)

df_long_all = pd.DataFrame(long_records)

In [22]:
# 6. Filter & count removals (missing, zero, negative yield)
removal_stats = []
df_cleaned_list = []

for crop in ['Rice', 'Maize', 'Cotton']:
    crop_df = df_long_all[df_long_all['crop'] == crop].copy()

    total_before = len(crop_df)
    missing_mask = crop_df['yield_kg_per_ha'].isnull()
    zero_mask = crop_df['yield_kg_per_ha'] == 0
    negative_mask = crop_df['yield_kg_per_ha'] < 0

    missing_count = missing_mask.sum()
    zero_count = zero_mask.sum()
    negative_count = negative_mask.sum()

    # Keep positive non-missing
    keep_mask = crop_df['yield_kg_per_ha'] > 0
    cleaned_crop_df = crop_df[keep_mask].copy()

    df_cleaned_list.append(cleaned_crop_df)

    removal_stats.append({
        'Crop': crop,
        'Initial': total_before,
        'Removed_Missing': missing_count,
        'Removed_Zero': zero_count,
        'Removed_Negative': negative_count,
        'Retained': len(cleaned_crop_df)
    })

df_cleaned = pd.concat(df_cleaned_list, ignore_index=True)
removals_df = pd.DataFrame(removal_stats)
print("Removal Breakdown per Crop:")
display(removals_df)

Removal Breakdown per Crop:


,Crop,Initial,Removed_Missing,Removed_Zero,Removed_Negative,Retained
0,Rice,16146,0,1143,22,14981
1,Maize,16146,0,1964,28,14154
2,Cotton,16146,0,8315,15,7816


In [23]:
# 7. & 8. Check exact duplicate rows and duplicate keys
exact_dups = df_cleaned.duplicated().sum()
print(f"Number of exact duplicate rows: {exact_dups}")

# Remove identical duplicate rows
df_cleaned = df_cleaned.drop_duplicates().copy()

# Check for duplicate [district_code, year, crop] keys (conflict analysis)
key_cols = ['district_code', 'year', 'crop']
all_key_duplicates = df_cleaned[df_cleaned.duplicated(subset=key_cols, keep=False)]

# Separate completely conflicting keys
conflicting_records = pd.DataFrame(columns=df_cleaned.columns)
if not all_key_duplicates.empty:
    # Find rows where values differ but keys are identical
    grouped = all_key_duplicates.groupby(key_cols)
    conflicting_indices = []
    for keys, group in grouped:
        # If any of the variables differ within the grouped keys
        if group[['area_1000_ha', 'production_1000_tons', 'yield_kg_per_ha']].nunique().max() > 1:
            conflicting_indices.extend(group.index.tolist())

    if conflicting_indices:
        conflicting_records = df_cleaned.loc[conflicting_indices].copy()
        # Remove conflicting duplicates from our clean interim dataset
        df_cleaned = df_cleaned.drop(index=conflicting_indices).copy()

conflicting_count = len(conflicting_records)
print(f"Number of conflicting-key records isolated: {conflicting_count}")

Number of exact duplicate rows: 0
Number of conflicting-key records isolated: 0


In [24]:
# 9, 10, 11 & 12. Yield Consistency calculation
# calculated_yield = production_1000_tons / area_1000_ha * 1000
# Checked only where area and production are positive (> 0)

calc_mask = (df_cleaned['area_1000_ha'] > 0) & (df_cleaned['production_1000_tons'] > 0)

df_cleaned['yield_consistency_flag'] = True
df_cleaned['percentage_difference'] = 0.0

if calc_mask.any():
    calculated_yield = (df_cleaned.loc[calc_mask, 'production_1000_tons'] / df_cleaned.loc[calc_mask, 'area_1000_ha']) * 1000
    reported_yield = df_cleaned.loc[calc_mask, 'yield_kg_per_ha']

    # Calculate percentage difference relative to reported yield (prevent divide by zero)
    pct_diff = np.where(
        reported_yield != 0,
        np.abs(reported_yield - calculated_yield) / reported_yield * 100,
        0.0
    )

    df_cleaned.loc[calc_mask, 'percentage_difference'] = pct_diff
    # Set flag to False if difference exceeds 10% tolerance limit
    df_cleaned.loc[calc_mask, 'yield_consistency_flag'] = pct_diff <= 10.0

flag_counts = df_cleaned['yield_consistency_flag'].value_counts()
print("Yield Consistency Flag Counts (True = within 10% tolerance, False = inconsistent):")
print(flag_counts)

Yield Consistency Flag Counts (True = within 10% tolerance, False = inconsistent):
yield_consistency_flag
True     36844
False      107
Name: count, dtype: int64


In [25]:
# 13, 14, 15. Save outputs
os.makedirs(os.path.dirname(interim_path), exist_ok=True)
os.makedirs(os.path.dirname(conflicting_path), exist_ok=True)
os.makedirs(os.path.dirname(report_path), exist_ok=True)

df_cleaned.to_csv(interim_path, index=False)
conflicting_records.to_csv(conflicting_path, index=False)

# Generate restructuring report summary text
report_txt = f"""FarmSight XAI Multicrop Dataset Restructuring Report
=======================================================
Source Dataset: {raw_path}
Target Crops: Rice, Maize, Cotton

Quality Control Summary:
-----------------------
Raw File Dimensions: {raw_shape[0]} rows, {raw_shape[1]} columns
Final Cleaned Interim Table Dimensions: {df_cleaned.shape[0]} rows, {df_cleaned.shape[1]} columns
Exact Identical Rows Removed: {exact_dups}
Conflicting Duplicate Key Records Removed: {conflicting_count}

Removal Breakdown by Reason & Crop:
-----------------------------------
{removals_df.to_string(index=False)}

Yield Consistency Flag Counts (Tolerance 10%):
---------------------------------------------
{flag_counts.to_string()}
"""

with open(report_path, 'w') as f:
    f.write(report_txt)

print(f"Saved long interim table to: {interim_path}")
print(f"Saved conflicting duplicate keys table to: {conflicting_path}")
print(f"Saved detailed restructuring log to: {report_path}")

Saved long interim table to: /content/drive/MyDrive/FarmSight_XAI/data/interim/icrisat_rice_maize_cotton_long.csv
Saved conflicting duplicate keys table to: /content/drive/MyDrive/FarmSight_XAI/results/tables/icrisat_conflicting_duplicates.csv
Saved detailed restructuring log to: /content/drive/MyDrive/FarmSight_XAI/logs/icrisat_restructuring_report.txt


In [26]:
# Print requested diagnostic details
print("\n--- Interim Restructuring Metrics ---")
print(f"Raw dataset shape: {raw_shape}")
print(f"Final interim dataset shape: {df_cleaned.shape}")
print(f"Exact Duplicate count: {exact_dups}")
print(f"Conflicting duplicate key count: {conflicting_count}")
print(f"Yield consistency flag distribution:\n{df_cleaned['yield_consistency_flag'].value_counts()}")

print("\nRows and Year Range per Crop in final interim table:")
for crop in ['Rice', 'Maize', 'Cotton']:
    subset = df_cleaned[df_cleaned['crop'] == crop]
    print(f"- {crop}: {len(subset)} rows, Year range: {subset['year'].min()} to {subset['year'].max()}")


--- Interim Restructuring Metrics ---
Raw dataset shape: (16146, 80)
Final interim dataset shape: (36951, 11)
Exact Duplicate count: 0
Conflicting duplicate key count: 0
Yield consistency flag distribution:
yield_consistency_flag
True     36844
False      107
Name: count, dtype: int64

Rows and Year Range per Crop in final interim table:
- Rice: 14981 rows, Year range: 1966 to 2017
- Maize: 14154 rows, Year range: 1966 to 2017
- Cotton: 7816 rows, Year range: 1966 to 2017


In [27]:
# --- Verification Suite ---
raw_hash_now = get_sha256(raw_path)
hash_valid = (raw_hash_now == expected_raw_hash)
files_exist = os.path.exists(interim_path) and os.path.exists(conflicting_path) and os.path.exists(report_path)

crops_in_table = df_cleaned['crop'].unique().tolist()
only_selected_crops = set(crops_in_table).issubset({'Rice', 'Maize', 'Cotton'})

yield_positive = (df_cleaned['yield_kg_per_ha'] > 0).all()
yield_not_null = df_cleaned['yield_kg_per_ha'].notnull().all()

print("\n--- Verification Status ---")
print(f"Raw file hash unchanged: {hash_valid}")
print(f"All three output files exist: {files_exist}")
print(f"Contains only Rice, Maize, Cotton: {only_selected_crops} ({crops_in_table})")
print(f"All yield values are positive: {yield_positive}")
print(f"All yield values are non-null: {yield_not_null}")

if hash_valid and files_exist and only_selected_crops and yield_positive and yield_not_null:
    print("\nMULTICROP INTERIM TABLE VERIFIED")
else:
    print("\nVERIFICATION FAILED")


--- Verification Status ---
Raw file hash unchanged: True
All three output files exist: True
Contains only Rice, Maize, Cotton: True (['Rice', 'Maize', 'Cotton'])
All yield values are positive: True
All yield values are non-null: True

MULTICROP INTERIM TABLE VERIFIED


In [28]:
import os
import re
import time
import hashlib
import requests
import pandas as pd
from difflib import SequenceMatcher

input_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/icrisat_rice_maize_cotton_long.csv"
accepted_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/district_coordinates_accepted.csv"
review_path = "/content/drive/MyDrive/FarmSight_XAI/results/tables/district_geocoding_review.csv"
log_path = "/content/drive/MyDrive/FarmSight_XAI/logs/district_geocoding_report.txt"

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def normalize(value):
    value = str(value).lower().strip()
    value = re.sub(r"\bdistrict\b", "", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())

state_aliases = {
    "orissa": "odisha",
    "uttaranchal": "uttarakhand",
    "pondicherry": "puducherry",
}

def similarity(a, b):
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()

raw_hash_before = file_sha256(input_path)
data = pd.read_csv(input_path)

districts = (
    data[
        ["district_code", "district_name", "state_code", "state_name"]
    ]
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
    .reset_index(drop=True)
)

records = []
endpoint = "https://geocoding-api.open-meteo.com/v1/search"

for index, row in districts.iterrows():
    district = str(row["district_name"]).strip()
    state_original = str(row["state_name"]).strip()
    state_query = state_aliases.get(state_original.lower(), state_original)

    queries = [f"{district}, {state_query}", district]
    candidates = []
    error_message = ""

    for query in queries:
        try:
            response = requests.get(
                endpoint,
                params={
                    "name": query,
                    "count": 10,
                    "language": "en",
                    "countryCode": "IN",
                    "format": "json",
                },
                timeout=30,
            )
            response.raise_for_status()
            candidates = response.json().get("results", [])
            if candidates:
                break
        except Exception as error:
            error_message = str(error)

        time.sleep(0.15)

    scored = []
    for candidate in candidates:
        location_fields = [
            candidate.get("name", ""),
            candidate.get("admin2", ""),
            candidate.get("admin3", ""),
            candidate.get("admin4", ""),
        ]

        district_score = max(
            similarity(district, value)
            for value in location_fields
            if value
        )

        state_score = similarity(
            state_query,
            candidate.get("admin1", "")
        )

        combined_score = (0.80 * district_score) + (0.20 * state_score)
        scored.append((combined_score, district_score, state_score, candidate))

    if scored:
        scored.sort(key=lambda item: item[0], reverse=True)
        combined_score, district_score, state_score, best = scored[0]

        accepted = (
            district_score >= 0.85
            and state_score >= 0.60
            and best.get("latitude") is not None
            and best.get("longitude") is not None
        )

        record = {
            **row.to_dict(),
            "matched_name": best.get("name"),
            "matched_admin1": best.get("admin1"),
            "matched_admin2": best.get("admin2"),
            "matched_admin3": best.get("admin3"),
            "latitude": best.get("latitude"),
            "longitude": best.get("longitude"),
            "timezone": best.get("timezone"),
            "feature_code": best.get("feature_code"),
            "district_score": round(district_score, 4),
            "state_score": round(state_score, 4),
            "combined_score": round(combined_score, 4),
            "candidate_count": len(candidates),
            "match_status": "accepted" if accepted else "needs_review",
            "error": error_message,
        }
    else:
        record = {
            **row.to_dict(),
            "matched_name": None,
            "matched_admin1": None,
            "matched_admin2": None,
            "matched_admin3": None,
            "latitude": None,
            "longitude": None,
            "timezone": None,
            "feature_code": None,
            "district_score": 0,
            "state_score": 0,
            "combined_score": 0,
            "candidate_count": 0,
            "match_status": "unresolved",
            "error": error_message,
        }

    records.append(record)

    if (index + 1) % 25 == 0 or index + 1 == len(districts):
        print(f"Processed {index + 1}/{len(districts)} districts")

    time.sleep(0.15)

geocoded = pd.DataFrame(records)
accepted = geocoded[geocoded["match_status"] == "accepted"].copy()
review = geocoded[geocoded["match_status"] != "accepted"].copy()

accepted.to_csv(accepted_path, index=False)
review.to_csv(review_path, index=False)

raw_hash_after = file_sha256(input_path)
raw_unchanged = raw_hash_before == raw_hash_after

summary = f"""
District records attempted: {len(districts)}
Accepted automatically: {len(accepted)}
Needs review or unresolved: {len(review)}
Raw interim file unchanged: {raw_unchanged}
Accepted file: {accepted_path}
Review file: {review_path}

Coordinate meaning:
Open-Meteo geocoded representative locations are used.
They are not claimed to be official district centroids.
"""

with open(log_path, "w", encoding="utf-8") as file:
    file.write(summary.strip() + "\n")

outputs_exist = all(
    os.path.exists(path) and os.path.getsize(path) > 0
    for path in [accepted_path, review_path, log_path]
)

print(summary)
print("Output files created:", outputs_exist)

if raw_unchanged and outputs_exist and len(accepted) + len(review) == len(districts):
    print("DISTRICT GEOCODING STAGE VERIFIED")
else:
    print("DISTRICT GEOCODING STAGE FAILED")

Processed 25/311 districts
Processed 50/311 districts
Processed 75/311 districts
Processed 100/311 districts
Processed 125/311 districts
Processed 150/311 districts
Processed 175/311 districts
Processed 200/311 districts
Processed 225/311 districts
Processed 250/311 districts
Processed 275/311 districts
Processed 300/311 districts
Processed 311/311 districts

District records attempted: 311
Accepted automatically: 229
Needs review or unresolved: 82
Raw interim file unchanged: True
Accepted file: /content/drive/MyDrive/FarmSight_XAI/data/interim/district_coordinates_accepted.csv
Review file: /content/drive/MyDrive/FarmSight_XAI/results/tables/district_geocoding_review.csv

Coordinate meaning:
Open-Meteo geocoded representative locations are used.
They are not claimed to be official district centroids.

Output files created: True
DISTRICT GEOCODING STAGE VERIFIED


In [29]:
import pandas as pd

long_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/icrisat_rice_maize_cotton_long.csv"
coords_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/district_coordinates_accepted.csv"

agri_df = pd.read_csv(long_path)
coords_df = pd.read_csv(coords_path)

print("Agricultural table shape:", agri_df.shape)
print("Agricultural columns:", agri_df.columns.tolist())

print("\nAccepted coordinates shape:", coords_df.shape)
print("Coordinate columns:", coords_df.columns.tolist())

Agricultural table shape: (36951, 11)
Agricultural columns: ['district_code', 'district_name', 'state_code', 'state_name', 'year', 'crop', 'area_1000_ha', 'production_1000_tons', 'yield_kg_per_ha', 'yield_consistency_flag', 'percentage_difference']

Accepted coordinates shape: (229, 18)
Coordinate columns: ['district_code', 'district_name', 'state_code', 'state_name', 'matched_name', 'matched_admin1', 'matched_admin2', 'matched_admin3', 'latitude', 'longitude', 'timezone', 'feature_code', 'district_score', 'state_score', 'combined_score', 'candidate_count', 'match_status', 'error']


In [30]:
from pathlib import Path
import hashlib
import pandas as pd

base = Path("/content/drive/MyDrive/FarmSight_XAI")
long_path = base / "data/interim/icrisat_rice_maize_cotton_long.csv"
coords_path = base / "data/interim/district_coordinates_accepted.csv"

crop_output = base / "results/tables/geocoding_coverage_by_crop.csv"
state_output = base / "results/tables/geocoding_coverage_by_state.csv"
log_output = base / "logs/geocoding_coverage_report.txt"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

hash_before = sha256_file(long_path)

agri = pd.read_csv(long_path)
coords = pd.read_csv(coords_path)

join_keys = ["state_code", "district_code"]

# Standardize ID types without modifying saved source files.
for column in join_keys:
    agri[column] = pd.to_numeric(agri[column], errors="raise").astype("Int64")
    coords[column] = pd.to_numeric(coords[column], errors="raise").astype("Int64")

duplicate_coordinate_keys = int(coords.duplicated(join_keys).sum())

coord_keys = coords[join_keys].drop_duplicates().copy()
coord_keys["coordinate_available"] = True

merged = agri.merge(
    coord_keys,
    on=join_keys,
    how="left",
    validate="many_to_one"
)

merged["coordinate_available"] = (
    merged["coordinate_available"].fillna(False).astype(bool)
)

def coverage_summary(group):
    total_rows = len(group)
    covered_rows = int(group["coordinate_available"].sum())

    total_districts = len(group[join_keys].drop_duplicates())
    covered_districts = len(
        group.loc[group["coordinate_available"], join_keys].drop_duplicates()
    )

    return {
        "total_rows": total_rows,
        "covered_rows": covered_rows,
        "row_coverage_pct": round(100 * covered_rows / total_rows, 2),
        "total_districts": total_districts,
        "covered_districts": covered_districts,
        "district_coverage_pct": round(
            100 * covered_districts / total_districts, 2
        ),
    }

crop_rows = []
for crop, group in merged.groupby("crop", dropna=False):
    crop_rows.append({"crop": crop, **coverage_summary(group)})

crop_coverage = pd.DataFrame(crop_rows).sort_values("crop")

state_rows = []
for (state_code, state_name), group in merged.groupby(
    ["state_code", "state_name"], dropna=False
):
    state_rows.append({
        "state_code": state_code,
        "state_name": state_name,
        **coverage_summary(group)
    })

state_coverage = (
    pd.DataFrame(state_rows)
    .sort_values(["district_coverage_pct", "state_name"])
)

overall = coverage_summary(merged)
states_below_70 = state_coverage[
    state_coverage["district_coverage_pct"] < 70
]

crop_output.parent.mkdir(parents=True, exist_ok=True)
log_output.parent.mkdir(parents=True, exist_ok=True)

crop_coverage.to_csv(crop_output, index=False)
state_coverage.to_csv(state_output, index=False)

hash_after = sha256_file(long_path)

report = f"""FarmSight XAI Geocoding Coverage Audit
Agricultural rows: {len(agri)}
Accepted coordinate rows: {len(coords)}
Duplicate coordinate keys: {duplicate_coordinate_keys}

Overall covered agricultural rows: {overall['covered_rows']}
Overall row coverage (%): {overall['row_coverage_pct']}
Overall covered districts: {overall['covered_districts']}
Overall districts: {overall['total_districts']}
Overall district coverage (%): {overall['district_coverage_pct']}

States below 70% district coverage: {len(states_below_70)}
Agricultural file hash unchanged: {hash_before == hash_after}

Coordinate interpretation:
Open-Meteo geocoded representative locations are used.
They are not claimed to be official district centroids.
"""

log_output.write_text(report, encoding="utf-8")

files_exist = all(
    path.exists() for path in [crop_output, state_output, log_output]
)

verified = (
    len(merged) == len(agri)
    and duplicate_coordinate_keys == 0
    and overall["covered_rows"] > 0
    and hash_before == hash_after
    and files_exist
)

print("Overall coverage:")
print(pd.Series(overall).to_string())

print("\nCoverage by crop:")
display(crop_coverage)

print("\nStates below 70% district coverage:")
display(states_below_70)

print("\nOutput files created:", files_exist)
print("Agricultural file hash unchanged:", hash_before == hash_after)
print("Duplicate coordinate keys:", duplicate_coordinate_keys)

if verified:
    print("\nGEOCODING COVERAGE AUDIT VERIFIED")
else:
    print("\nGEOCODING COVERAGE AUDIT FAILED")

Overall coverage:
total_rows               36951.00
covered_rows             27026.00
row_coverage_pct            73.14
total_districts            311.00
covered_districts          229.00
district_coverage_pct       73.63

Coverage by crop:


/tmp/ipykernel_607/945274795.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged["coordinate_available"].fillna(False).astype(bool)


,crop,total_rows,covered_rows,row_coverage_pct,total_districts,covered_districts,district_coverage_pct
0,Cotton,7816,5459,69.84,231,167,72.29
1,Maize,14154,10567,74.66,302,221,73.18
2,Rice,14981,11000,73.43,307,227,73.94



States below 70% district coverage:


,state_code,state_name,total_rows,covered_rows,row_coverage_pct,total_districts,covered_districts,district_coverage_pct
10,11,Tamil Nadu,1558,623,39.99,12,4,33.33
0,1,Andhra Pradesh,1571,679,43.22,11,5,45.45
4,5,Karnataka,2619,1303,49.75,19,9,47.37
16,17,Assam,1301,595,45.73,10,5,50.00
14,15,Jharkhand,597,298,49.92,6,3,50.00
15,16,Uttarakhand,676,364,53.85,8,4,50.00
7,8,Orissa,1741,941,54.05,13,7,53.85
2,3,Gujarat,2335,1571,67.28,18,12,66.67
12,13,West Bengal,1498,975,65.09,15,10,66.67



Output files created: True
Agricultural file hash unchanged: True
Duplicate coordinate keys: 0

GEOCODING COVERAGE AUDIT VERIFIED
